# Introductino to Computer Vision (ECSE 415)

## Assignment 5: Video Analysis

DEADLINE: December 3rd, 11:59 PM
Group: 21

Name: Steve Chen
ID: 261106847

## 0. Setup and Environment

In [66]:
from pathlib import Path
import cv2  # or PIL.Image
import os

# Base directory
base_dir = Path("Object_Tracking")

# Task1 paths
task1_images = base_dir / "Task1" / "images"
task1_gt = base_dir / "Task1" / "gt" / "gt.txt"

# Task2 paths
task2_images = base_dir / "Task2" / "images"

print("Base directory and task paths have been set up.")
print(f"Task 1 images path: {task1_images}")
print(f"Task 1 ground truth path: {task1_gt}")
print(f"Task 2 images path: {task2_images}")

Base directory and task paths have been set up.
Task 1 images path: Object_Tracking\Task1\images
Task 1 ground truth path: Object_Tracking\Task1\gt\gt.txt
Task 2 images path: Object_Tracking\Task2\images


## 1. Data Preparation

In [58]:
# Video Name
output_video = "task1_input.mp4"

# Get all image files, sorted (important for correct frame order)
images = sorted([img for img in os.listdir(task1_images) if img.endswith((".jpg", ".png", ".jpeg"))])

# Read the first image to get frame size
first_frame = cv2.imread(os.path.join(task1_images, images[0]))
height, width, layers = first_frame.shape

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 'mp4v' works well for .mp4
video = cv2.VideoWriter(output_video, fourcc, 14, (width, height))

# Write each image as a frame
for image in images:
    frame = cv2.imread(os.path.join(task1_images, image))
    video.write(frame)

# Release the video writer
video.release()

print(f"Video saved as {output_video}")

Video saved as task1_input.mp4


## 2. Model Implementation

In [59]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot.trackers.strongsort.strongsort import StrongSort
import torch

# ---------------- DEVICE CHECK ----------------
if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    device = "cuda:0"
    half = True
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    half = False
    print("⚠️ No GPU detected. Using CPU instead.")

# ---------------- CONFIGURATION ----------------
input_video = "task1_input.mp4"
output_video = "task1_strongsort.mp4"
results_txt = "task1_strongsort.txt"
model_path = "yolov8m.pt"
reid_weights = Path("osnet_x0_25_msmt17.pt")  # Path object required by StrongSort

# ---------------- INITIALIZATION ----------------
print("🚀 Loading YOLOv8 model...")
model = YOLO(model_path)

print("⚙️ Setting up StrongSORT...")
tracker = StrongSort(
    reid_weights=reid_weights,
    device=device,
    half=half,
    max_age=30,   # longer memory
    det_thresh=0.002
)

# ---------------- VIDEO SETUP ----------------
cap = cv2.VideoCapture(input_video)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

frame_idx = 0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"▶️ Processing {input_video} ({total_frames} frames)...")

# ---------------- PROCESS FRAMES ----------------
with open(results_txt, "w") as f:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ---------------- DETECTION ----------------
        results = model.predict(frame, conf=0.002, classes=[0], verbose=False, device=device)[0]

        if hasattr(results, "boxes") and results.boxes is not None and results.boxes.data is not None:
            dets = results.boxes.data.cpu().numpy()
        else:
            dets = np.empty((0, 6))

        # ---------------- TRACKING ----------------
        tracks = tracker.update(dets, frame)

        # ---------------- DRAWING & SAVING ----------------
        for track in tracks:
            x1, y1, x2, y2 = map(int, track[:4])
            track_id = int(track[4])
            w = x2 - x1
            h = y2 - y1

            # Draw bounding box and ID
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Write tracking info to file
            f.write(f"{frame_idx}, {track_id}, {x1}, {y1}, {w}, {h}\n")

        out.write(frame)

        # Print progress every 20 frames
        if frame_idx % 20 == 0:
            print(f"Processing frame {frame_idx}/{total_frames}", end="\r")

        frame_idx += 1

# ---------------- CLEAN UP ----------------
cap.release()
out.release()
print(f"\n✅ Tracking complete. Saved to {output_video}")


2025-11-29 00:31:32.234 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:56 | __init__ - BaseTracker initialization parameters:
2025-11-29 00:31:32.234 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:57 | __init__ - det_thresh: 0.002
2025-11-29 00:31:32.234 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:58 | __init__ - max_age: 30
2025-11-29 00:31:32.234 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:59 | __init__ - max_obs: 50
2025-11-29 00:31:32.236 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:60 | __init__ - min_hits:

⚠️ No GPU detected. Using CPU instead.
🚀 Loading YOLOv8 model...
⚙️ Setting up StrongSORT...
▶️ Processing task1_input.mp4 (429 frames)...
Processing frame 420/429
✅ Tracking complete. Saved to task1_strongsort.mp4


In [60]:
# import cv2
# from ultralytics import YOLO
# from deep_sort_realtime.deepsort_tracker import DeepSort

# # Load YOLOv8 model (pretrained on COCO)
# model = YOLO("yolov8m.pt")  # you can use yolov8s.pt or larger models for better accuracy

# # Initialize DeepSORT tracker
# tracker = DeepSort(max_age=30, n_init=3, nms_max_overlap=1.0, max_cosine_distance=0.5, max_iou_distance=0.85)

# # Input and output video paths
# input_video = "task1_input.mp4"
# output_video = "task1.mp4"
# results_txt = "task1.txt"

# # Open video
# cap = cv2.VideoCapture(input_video)
# fps = int(cap.get(cv2.CAP_PROP_FPS))
# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# # Video writer
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# frame_idx = 0
# with open(results_txt, "w") as f:
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Run YOLOv8 detection (only 'person' class, class_id=0 in COCO)
#         results = model(frame)[0]
#         detections = []
#         for box in results.boxes:
#             cls = int(box.cls[0])
#             if cls == 0:  # person
#                 x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
#                 conf = float(box.conf[0])
#                 detections.append(([x1, y1, x2 - x1, y2 - y1], conf, cls))

#         # Update DeepSORT tracker
#         tracks = tracker.update_tracks(detections, frame=frame)

#         # Draw bounding boxes and save results
#         for track in tracks:
#             if not track.is_confirmed():
#                 continue
#             track_id = track.track_id
#             ltrb = track.to_ltrb()  # left, top, right, bottom
#             x1, y1, x2, y2 = map(int, ltrb)
#             w, h = x2 - x1, y2 - y1

#             # Draw box + ID
#             cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#             cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

#             # Save tracking results in required format
#             f.write(f"{frame_idx}, {track_id}, {x1}, {y1}, {w}, {h}\n")

#         out.write(frame)
#         frame_idx += 1

# cap.release()
# out.release()
# print("✅ Tracking complete. Video saved as task1.mp4 and results in task1.txt")

## 3. Model Evaluation

In [61]:
from collections import defaultdict
import numpy as np

# Hungarian algorithm
try:
    from scipy.optimize import linear_sum_assignment
except ImportError:
    raise ImportError("Please install SciPy: pip install scipy")

GT_PATH = "Object_Tracking/Task1/gt/gt.txt"
PRED_PATH = "task1.txt"
PRED_PATH = "task1_strongsort.txt"
IOU_THRESH = 0.5

def iou_xywh(a, b):
    """
    a, b: [x, y, w, h]
    """
    ax1, ay1 = a[0], a[1]
    ax2, ay2 = a[0] + a[2], a[1] + a[3]
    bx1, by1 = b[0], b[1]
    bx2, by2 = b[0] + b[2], b[1] + b[3]

    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    a_area = max(0.0, (ax2 - ax1)) * max(0.0, (ay2 - ay1))
    b_area = max(0.0, (bx2 - bx1)) * max(0.0, (by2 - by1))

    denom = a_area + b_area - inter_area
    if denom <= 0:
        return 0.0
    return inter_area / denom

def load_gt(path):
    """
    Returns dict: frame -> dict(gt_id -> [x,y,w,h])
    Assumes MOTChallenge-like CSV with at least 6 columns:
    frame,id,x,y,w,h,...
    """
    gt_by_frame = defaultdict(dict)
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            # tolerate spaces
            parts = [p.strip() for p in parts]
            if len(parts) < 6:
                # skip malformed lines
                continue
            frame = int(parts[0])
            gid = int(parts[1])
            x, y, w, h = map(float, parts[2:6])
            gt_by_frame[frame][gid] = [x, y, w, h]
    return gt_by_frame

def load_pred(path):
    """
    Returns dict: frame -> dict(pred_id -> [x,y,w,h])
    Expects: frame,id,x,y,w,h
    """
    pred_by_frame = defaultdict(dict)
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            parts = [p.strip() for p in parts]
            if len(parts) < 6:
                continue
            frame = int(parts[0])
            pid = int(parts[1])
            x, y, w, h = map(float, parts[2:6])
            pred_by_frame[frame][pid] = [x, y, w, h]
    return pred_by_frame

def match_frame(gt_boxes, pred_boxes, iou_thresh=IOU_THRESH):
    """
    gt_boxes: dict(gt_id -> [x,y,w,h])
    pred_boxes: dict(pred_id -> [x,y,w,h])
    Returns:
      matches: list of (gt_id, pred_id)
      fp_ids: list of pred_ids with no match
      fn_ids: list of gt_ids with no match
    """
    gt_ids = list(gt_boxes.keys())
    pred_ids = list(pred_boxes.keys())

    if len(gt_ids) == 0 and len(pred_ids) == 0:
        return [], [], []

    if len(gt_ids) == 0:
        return [], pred_ids, []
    if len(pred_ids) == 0:
        return [], [], gt_ids

    # Build IoU matrix: rows=gt, cols=pred
    iou_mat = np.zeros((len(gt_ids), len(pred_ids)), dtype=np.float32)
    for i, gid in enumerate(gt_ids):
        for j, pid in enumerate(pred_ids):
            iou_mat[i, j] = iou_xywh(gt_boxes[gid], pred_boxes[pid])

    # Hungarian over cost = 1 - IoU
    cost = 1.0 - iou_mat
    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    matched_gt = set()
    matched_pred = set()

    for r, c in zip(row_ind, col_ind):
        iou = iou_mat[r, c]
        if iou >= iou_thresh:
            gid = gt_ids[r]
            pid = pred_ids[c]
            matches.append((gid, pid))
            matched_gt.add(gid)
            matched_pred.add(pid)

    fn_ids = [gid for gid in gt_ids if gid not in matched_gt]
    fp_ids = [pid for pid in pred_ids if pid not in matched_pred]

    return matches, fp_ids, fn_ids

def compute_mota(gt_by_frame, pred_by_frame):
    """
    Tracks FP, FN, IDSW per frame and computes MOTA.
    Identity switches are counted when an ongoing GT id
    is matched to a different pred id than in the previous matched frame.
    """
    frames = sorted(set(gt_by_frame.keys()) | set(pred_by_frame.keys()))
    total_FP = 0
    total_FN = 0
    total_IDSW = 0
    total_GT = 0

    # For ID switches: map gt_id -> last matched pred_id
    last_match = {}

    for t in frames:
        gt_boxes = gt_by_frame.get(t, {})
        pred_boxes = pred_by_frame.get(t, {})

        total_GT += len(gt_boxes)

        matches, fp_ids, fn_ids = match_frame(gt_boxes, pred_boxes, IOU_THRESH)
        total_FP += len(fp_ids)
        total_FN += len(fn_ids)

        # Count ID switches
        # Only when the same GT id was previously matched and now matched to a different pred id
        for gid, pid in matches:
            if gid in last_match:
                if last_match[gid] != pid:
                    total_IDSW += 1
            # update last matched id
            last_match[gid] = pid

        # If a GT id is unmatched this frame, do not clear last_match;
        # IDSW only triggers when GT is matched again to a different pred id later.

    mota = 1.0
    if total_GT > 0:
        mota = 1.0 - (total_FN + total_FP + total_IDSW) / float(total_GT)

    return {
        "MOTA": mota,
        "FP": total_FP,
        "FN": total_FN,
        "IDSW": total_IDSW,
        "GT": total_GT
    }

gt_by_frame = load_gt(GT_PATH)
pred_by_frame = load_pred(PRED_PATH)
stats = compute_mota(gt_by_frame, pred_by_frame)

print("MOTA evaluation (IoU ≥ 0.5, Hungarian assignment)")
print(f"MOTA: {stats['MOTA']:.4f}")
print(f"Breakdown:")
print(f" - FP:   {stats['FP']}")
print(f" - FN:   {stats['FN']}")
print(f" - IDSW: {stats['IDSW']}")
print(f" - GT:   {stats['GT']}")

MOTA evaluation (IoU ≥ 0.5, Hungarian assignment)
MOTA: 0.3471
Breakdown:
 - FP:   985
 - FN:   16057
 - IDSW: 357
 - GT:   26647


## 4.Prediction & Kaggle Competition

In [67]:
import os
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot.trackers.strongsort.strongsort import StrongSort
import torch
import csv

# ---------------- DEVICE CHECK ----------------
if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    device = "cuda:0"
    half = True
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    half = False
    print("⚠️ No GPU detected. Using CPU instead.")

# ---------------- VIDEO CREATION FROM IMAGES ----------------
output_video_input = "task2_input.mp4"

images = sorted([img for img in os.listdir(task2_images) if img.endswith((".jpg", ".png", ".jpeg"))])
first_frame = cv2.imread(os.path.join(task2_images, images[0]))
height, width, layers = first_frame.shape

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_input, fourcc, 14, (width, height))

for image in images:
    frame = cv2.imread(os.path.join(task2_images, image))
    video_writer.write(frame)

video_writer.release()
print(f"✅ Video saved as {output_video_input}")

# ---------------- CONFIGURATION ----------------
input_video = output_video_input
output_video = "task2_strongsort.mp4"
results_txt = "task2_strongsort.txt"
counts_csv = "task2_counts.csv"
model_path = "yolov8m.pt"
reid_weights = Path("osnet_x0_25_msmt17.pt")

# ---------------- INITIALIZATION ----------------
model = YOLO(model_path)
tracker = StrongSort(
    reid_weights=reid_weights,
    device=device,
    half=half,
    max_age=30,
    det_thresh=0.002
)

# ---------------- VIDEO SETUP ----------------
cap = cv2.VideoCapture(input_video)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

frame_idx = 0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"▶️ Processing {input_video} ({total_frames} frames)...")

# ---------------- PROCESS FRAMES ----------------
ped_counts = []  # store frame_id -> count

with open(results_txt, "w") as f:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=0.002, classes=[0], verbose=False, device=device)[0]

        if hasattr(results, "boxes") and results.boxes is not None and results.boxes.data is not None:
            dets = results.boxes.data.cpu().numpy()
        else:
            dets = np.empty((0, 6))

        tracks = tracker.update(dets, frame)

        # Count people in this frame
        frame_count = len(tracks)
        ped_counts.append((frame_idx + 1, frame_count))  # frame IDs start from 1

        # Draw boxes & IDs
        for track in tracks:
            x1, y1, x2, y2 = map(int, track[:4])
            track_id = int(track[4])
            w = x2 - x1
            h = y2 - y1
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            f.write(f"{frame_idx},{track_id},{x1},{y1},{w},{h}\n")

        out.write(frame)

        if frame_idx % 20 == 0:
            print(f"Processing frame {frame_idx}/{total_frames}", end="\r")

        frame_idx += 1

# ---------------- SAVE PEDESTRIAN COUNTS ----------------
with open(counts_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Number", "Count"])
    writer.writerows(ped_counts)

# ---------------- CLEAN UP ----------------
cap.release()
out.release()
print(f"\n✅ Tracking complete. Annotated video saved as {output_video}")
print(f"✅ Pedestrian counts saved as {counts_csv}")


⚠️ No GPU detected. Using CPU instead.


2025-11-29 01:01:11.667 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:56 | __init__ - BaseTracker initialization parameters:
2025-11-29 01:01:11.668 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:57 | __init__ - det_thresh: 0.002
2025-11-29 01:01:11.668 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:58 | __init__ - max_age: 30
2025-11-29 01:01:11.668 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:59 | __init__ - max_obs: 50
2025-11-29 01:01:11.668 | MainProcess/MainThread | INFO     | c:\Users\songy\McGill\ComputerVisionMiscellaneous\venv\Lib\site-packages\boxmot\trackers\basetracker.py:60 | __init__ - min_hits:

✅ Video saved as task2_input.mp4
▶️ Processing task2_input.mp4 (1050 frames)...
Processing frame 1040/1050
✅ Tracking complete. Annotated video saved as task2_strongsort.mp4
✅ Pedestrian counts saved as task2_counts.csv


## 5. Deliverables